# LungXplain v5 — kiểm tra output và so sánh baseline

Chạy **sau khi** notebook native ACC-ECBM hoàn tất. Thêm private input
`ehehee/lungxplain-v5-private-input` và output private của notebook training
qua **Add Input → Notebook**, hoặc đặt các ZIP output trong input private.
Notebook này chỉ dùng CPU, không huấn luyện lại CNN.

Mọi code nằm trực tiếp trong cell, không cần source project. Dùng cùng cohort,
split và concept ngoài fold để fit clinical-only, concept-only và concept+clinical
với EBM/logistic. Chỉ phân tích validation; test tiếp tục được giữ kín.

Output MIMIC, kể cả dự đoán từng bệnh nhân, phải giữ private.

In [ ]:
import importlib.metadata
import subprocess
import sys
try:
    installed = importlib.metadata.version("interpret-core")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "0.7.8":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "interpret-core==0.7.8"])

import hashlib
import io
import json
import math
import os
import platform
import tarfile
import zipfile
from pathlib import Path, PurePosixPath
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from interpret.glassbox import ExplainableBoostingClassifier

INPUT = Path(os.environ.get("LX_ANALYSIS_INPUT", "/kaggle/input"))
OUTPUT = Path(os.environ.get("LX_ANALYSIS_OUTPUT", "/kaggle/working/lungxplain_analysis"))
SEARCH = [Path("/kaggle/working/lungxplain_native/runs"), INPUT]
SEEDS = [42, 123, 2026]
BOOTSTRAP = 2000
EXPECTED_MANIFEST = "ac3a6119f4267b31b874c2198990d33593bd6ffbcd1aae4943733bddcc46899a"
EXPECTED_BUNDLE = "8b6d121c4f867c4598c012bb1f9fee43e1411247f946e0c1efd8cb7fcd143b4a"
EXPECTED_CNN = {"protocol": "lungxplain-v5-native-1", "outer_folds": 3, "inner_fraction": 0.1,
                "max_epochs": 60, "min_epochs": 10, "patience": 10}
MAIN_VARIANT = "calibrated_intervention_aware"
VARIANTS = ["uncalibrated_standard_ebm", "uncalibrated_intervention_aware",
            "calibrated_standard_ebm", MAIN_VARIANT]
OUTPUT.mkdir(parents=True, exist_ok=True)

def hash_stream(stream):
    result = hashlib.sha256()
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        result.update(block)
    return result.hexdigest()

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hash_stream(stream)

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.ndarray):
        return clean_json(value.tolist())
    if isinstance(value, np.generic):
        return clean_json(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(clean_json(value), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

def require(condition, message):
    if not condition:
        raise ValueError(message)

## 1. Đọc đúng cohort v5

Chỉ cần manifest, schema và dataset validation; không đọc ảnh, nhãn bác sĩ hoặc
gói source ZIP. Clinical giữ đúng allowlist 44 cột. Sex được mã hóa bằng mapping
cố định trong schema; giá trị thiếu để NaN. Test không được đưa vào mảng mortality.

In [ ]:
def read_cohort(input_root):
    matches = [p for p in input_root.rglob("bundle_manifest_v5.json") if file_hash(p) == EXPECTED_BUNDLE]
    require(len(matches) == 1, "Attach the audited private v5 dataset exactly once.")
    bundle_path = matches[0]
    bundle = json.loads(bundle_path.read_text())
    root = bundle_path.parent
    names = [bundle["dataset_dir"] + "/" + name for name in
             ["manifest.csv", "feature_schema.json", "validation_report.json"]]
    contents = {}
    for name in names:
        options = [root / name, root / bundle.get("aliases", {}).get(name, name)]
        options += list(root.glob("**/" + name))
        for path in options:
            if path.is_file() and file_hash(path) == bundle["members"][name]:
                contents[name] = path.read_bytes()
                break
    if len(contents) != len(names):
        archives = list(root.rglob("lungxplain_v5_data.tar"))
        require(len(archives) == 1 and file_hash(archives[0]) == bundle["data_tar_sha256"],
                "Data archive checksum mismatch or metadata missing.")
        with tarfile.open(archives[0], "r:") as archive:
            for name in names:
                member = archive.getmember(name)
                require(member.isfile(), "Dataset metadata must be a regular file.")
                contents[name] = archive.extractfile(member).read()
    for name, content in contents.items():
        require(hashlib.sha256(content).hexdigest() == bundle["members"][name], "Metadata checksum failed.")
    frame = pd.read_csv(io.BytesIO(contents[names[0]]), low_memory=False)
    schema = json.loads(contents[names[1]])
    validation = json.loads(contents[names[2]])
    manifest_hash = hashlib.sha256(contents[names[0]]).hexdigest()
    require(manifest_hash == EXPECTED_MANIFEST and validation["passed"] is True and
            validation["exact_5000_validation"] == {"valid_images": 5000, "required": 5000}, "v5 validation failed.")
    require(len(frame) == 5000 and frame["subject_id"].nunique() == 5000 and
            frame["hadm_id"].nunique() == 5000 and frame["dicom_id"].nunique() == 5000, "Patient uniqueness failed.")
    require(frame["split"].value_counts().to_dict() == {"train": 4000, "valid": 500, "test": 500}, "Wrong split.")
    concepts, clinical = schema["concept_label_columns"], schema["model_clinical_columns"]
    require(len(concepts) == 14 and len(clinical) == 44, "Wrong model feature dimensions.")
    train = np.flatnonzero(frame["split"].eq("train"))
    valid = np.flatnonzero(frame["split"].eq("valid"))
    test = np.flatnonzero(frame["split"].eq("test"))
    y = np.full(len(frame), np.nan)
    y[np.r_[train, valid]] = frame.loc[np.r_[train, valid], schema["outcome_column"]].to_numpy(dtype=float)
    values = frame[clinical].copy()
    for column in clinical:
        if column in schema["categorical_value_columns"]:
            mapping = {v: i for i, v in enumerate(schema["categorical_values"][column])}
            require(frame[column].dropna().isin(mapping).all(), "Unknown clinical category.")
            values[column] = frame[column].map(mapping)
        else:
            values[column] = pd.to_numeric(values[column], errors="raise")
    types = ["continuous"] * 14 + ["nominal" if c.endswith("_missing") or
             c in schema["categorical_value_columns"] else "continuous" for c in clinical]
    return {"frame": frame, "schema": schema, "train": train, "valid": valid, "test": test,
            "y": y, "z": values.to_numpy(float), "labels": frame[concepts].to_numpy(float),
            "known": frame[["known_" + c for c in concepts]].to_numpy() == 1,
            "names": concepts + clinical, "types": types, "manifest_hash": manifest_hash,
            "schema_hash": hashlib.sha256(contents[names[1]]).hexdigest()}

class RunFiles:
    def __init__(self, path):
        self.path = Path(path)
        self.archive = zipfile.ZipFile(path) if self.path.is_file() else None
        self.prefix = ""
        if self.archive:
            names = [m.filename for m in self.archive.infolist() if not m.is_dir()]
            require(len(names) == len(set(names)), "Duplicate ZIP members.")
            require(all(not PurePosixPath(n).is_absolute() and ".." not in PurePosixPath(n).parts
                        and "\\" not in n for n in names), "Unsafe ZIP member.")
            complete = [name for name in names if name.count("/") == 1 and name.endswith("/complete.json")]
            require(len(complete) == 1, "ZIP must contain one completed seed.")
            self.prefix = complete[0].removesuffix("complete.json")
        self.stamp = self.json("complete.json")
        self.seed = int(self.stamp["seed"])

    def open(self, name):
        path = PurePosixPath(name)
        require(not path.is_absolute() and ".." not in path.parts and "\\" not in name, "Unsafe artifact path.")
        if self.archive:
            return self.archive.open(self.prefix + name)
        require(not (self.path / name).is_symlink(), "Symlink artifacts are not accepted.")
        return (self.path / name).open("rb")

    def read(self, name):
        with self.open(name) as stream:
            return stream.read()

    def json(self, name):
        return json.loads(self.read(name))

    def npz(self, name):
        with np.load(io.BytesIO(self.read(name)), allow_pickle=False) as saved:
            return {key: saved[key].copy() for key in saved.files}

    def model(self, name):
        require(name in self.stamp["artifacts"], "Model is absent from completed artifact manifest.")
        with self.open(name) as stream:
            require(hash_stream(stream) == self.stamp["artifacts"][name], "Model checksum failed.")
        return joblib.load(io.BytesIO(self.read(name)))

def find_runs(roots):
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates += [p.parent for p in root.rglob("complete.json") if p.parent.name in
                       {f"seed{seed}" for seed in SEEDS} and (p.parent / "binding.json").is_file()]
        candidates += list(root.rglob("lungxplain_native_seed*.zip"))
    chosen, identities = {}, {}
    for path in sorted(set(candidates), key=lambda p: (p.is_file(), str(p))):
        reader = RunFiles(path)
        if reader.seed not in SEEDS:
            continue
        identity = hashlib.sha256(reader.read("complete.json")).hexdigest()
        if reader.seed in chosen:
            require(identities[reader.seed] == identity, "Multiple different runs for a seed. Select one explicitly.")
        else:
            chosen[reader.seed], identities[reader.seed] = reader, identity
    return chosen

## 2. Kiểm tra artifact trước khi đọc metrics

Đối chiếu SHA-256 toàn bộ artifact, cohort/split, ba outer folds và inner split,
median epoch refit, feature allowlist và tái tạo dự đoán EBM. Chấm lại metrics
từ probabilities và validation labels. Audit này chưa chạy lại ảnh qua CNN và
chưa tái dựng quá trình fit Platt/alpha; hash/binding được giữ để audit sâu sau.

In [ ]:
def metric_values(y, p, threshold):
    return {"auroc": float(roc_auc_score(y, p)), "auprc": float(average_precision_score(y, p)),
            "macro_f1": float(f1_score(y, p >= threshold, average="macro", zero_division=0)),
            "brier": float(brier_score_loss(y, p)), "logloss": float(log_loss(y, p, labels=[0, 1]))}

def choose_threshold(y, p):
    candidates = np.unique(np.r_[0.0, p, np.nextafter(1.0, 2.0)])
    scores = [f1_score(y, p >= t, average="macro", zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

def audit_run(source, data, expected_cnn=None):
    expected_cnn = EXPECTED_CNN if expected_cnn is None else expected_cnn
    binding, stamp = source.json("binding.json"), source.stamp
    require(binding["manifest_sha256"] == data["manifest_hash"] and binding["schema_sha256"] == data["schema_hash"],
            "Run belongs to a different dataset/schema.")
    require(binding["seed"] == source.seed and binding["test_evaluated"] is False and stamp["test_evaluated"] is False,
            "Seed identity or test isolation failed.")
    config = binding["configuration"]
    for name, expected in expected_cnn.items():
        require(config.get(name) == expected, "CNN protocol differs: " + name)
    verified = 0
    for name, digest in stamp["artifacts"].items():
        with source.open(name) as stream:
            require(hash_stream(stream) == digest, "Artifact checksum failed.")
        verified += 1
    cached = source.npz("concept_predictions.npz")
    raw, folds = cached["probabilities"], cached["fold_id"]
    train, valid, test = [data[name] for name in ["train", "valid", "test"]]
    require(raw.shape == (len(data["frame"]), 14) and folds.shape == (len(raw),), "Prediction cache shape failed.")
    require(np.array_equal(cached["train_rows"], train) and np.array_equal(cached["valid_rows"], valid), "Row alignment failed.")
    require(np.isfinite(raw[np.r_[train, valid]]).all() and
            ((raw[np.r_[train, valid]] >= 0) & (raw[np.r_[train, valid]] <= 1)).all(), "Invalid concept probabilities.")
    require(np.isnan(raw[test]).all() and np.isnan(data["y"][test]).all(), "Test was accessed.")
    selected = []
    splitter = StratifiedKFold(config["outer_folds"], shuffle=True, random_state=source.seed)
    for fold, (development, heldout) in enumerate(splitter.split(train, data["y"][train])):
        require(np.array_equal(train[folds[train] == fold], np.sort(train[heldout])), "Outer OOF rows differ.")
        spec = source.json(f"fold{fold}/stage.json")
        fit_rows, inner_rows = train_test_split(train[development], test_size=config["inner_fraction"],
            random_state=source.seed + fold, stratify=data["y"][train[development]])
        require(spec["fitting_rows"] == fit_rows.tolist() and spec["selection_rows"] == inner_rows.tolist(), "Inner split differs.")
        history = source.json(f"fold{fold}/history.json")
        require(config["min_epochs"] <= len(history) <= config["max_epochs"] and
                [entry["epoch"] for entry in history] == list(range(1, len(history) + 1)) and
                history[-1]["optimizer_steps"] > 0 and spec["stage_seed"] == source.seed + fold,
                "CNN history incomplete or stage seed differs.")
        best, epoch = -math.inf, 0
        for entry in history:
            if entry["inner_concept_auroc"] > best + 1e-8:
                best, epoch = entry["inner_concept_auroc"], entry["epoch"]
        selected.append(epoch)
    require(set(folds[train]) == set(range(config["outer_folds"])) and (folds[np.r_[valid, test]] == -1).all(),
            "Fold coverage failed.")
    summary = source.json("cnn_summary.json")
    require(len(summary["folds"]) == config["outer_folds"] and
            [fold["best_epoch"] for fold in summary["folds"]] == selected,
            "CNN summary selected epochs differ from inner AUROC histories.")
    final_spec = source.json("final_cnn/stage.json")
    budget = max(1, int(np.median(selected)))
    require(summary["refit_epochs"] == budget and summary["test_predicted"] is False and
            final_spec["fitting_rows"] == train.tolist() and final_spec["selection_rows"] == [] and
            final_spec["refit_epochs"] == budget and len(source.json("final_cnn/history.json")) == budget,
            "Final CNN refit protocol failed.")
    records = source.json("validation_summary.json")
    require(len(records) == 4 and {r["variant"] for r in records} == set(VARIANTS), "Missing EBM variant.")
    observed_probabilities, validation_concepts = {}, {}
    y = data["y"][valid].astype(int)
    for record in records:
        variant = record["variant"]
        model = source.model(variant + "/ebm.joblib")
        require(list(model.feature_names_in_) == data["names"] and model.n_features_in_ == 58,
                "Outcome feature allowlist changed.")
        for name, expected in config["ebm"].items():
            require(model.get_params()[name] == expected, "EBM configuration differs from binding.")
        outcome = source.npz(variant + "/outcome_validation.npz")
        require(np.array_equal(outcome["rows"], valid), "Outcome prediction rows differ.")
        concepts = source.npz(variant + "/intervention_predictions_validation.npz")["probabilities"]
        require(concepts.shape == (len(valid), 14) and np.isfinite(concepts).all(), "Invalid intervention base cache.")
        if variant.startswith("uncalibrated"):
            require(np.array_equal(concepts, raw[valid]), "Uncalibrated concept cache differs.")
        replayed = model.predict_proba(np.column_stack([concepts, data["z"][valid]]))[:, 1]
        require(np.allclose(replayed, outcome["probabilities"], atol=1e-12, rtol=0), "EBM predictions do not reproduce.")
        threshold = source.json(variant + "/threshold_validation.json")
        require(threshold["selection_split"] == "valid" and threshold["test_evaluated"] is False and
                threshold["criterion"] == "macro_f1" and threshold["threshold"] == choose_threshold(y, replayed),
                "Threshold selection differs.")
        scores = metric_values(y, replayed, threshold["threshold"])
        require(all(np.isclose(scores[k], record[k], atol=1e-12, rtol=0) for k in scores), "Published metric differs from predictions.")
        curve = pd.read_csv(io.BytesIO(source.read(variant + "/interventions_validation.csv")))
        require(len(curve) == 45 and not curve.duplicated(["requested_concepts", "strategy"]).any(), "Incomplete intervention grid.")
        require(curve[curve.requested_concepts.eq(0)].auroc.nunique() == 1 and
                curve[curve.requested_concepts.eq(0)].logloss.nunique() == 1, "Zero-confirmation identity failed.")
        training_mask = source.npz(variant + "/training_intervention_mask.npz")["observed"]
        require(training_mask.shape == (len(train), 14) and not (training_mask & ~data["known"][train]).any(),
                "Training intervention used unknown reference labels.")
        if variant.endswith("standard_ebm"):
            require(not training_mask.any(), "Standard EBM contains training interventions.")
        observed_probabilities[variant] = replayed
        validation_concepts[variant] = concepts
    require(np.array_equal(validation_concepts[MAIN_VARIANT], validation_concepts["calibrated_standard_ebm"]),
            "Calibrated variants use different concept predictions.")
    return {"passed": True, "seed": source.seed, "verified_artifacts": verified,
            "refit_epochs": budget, "raw": raw, "outcomes": observed_probabilities, "records": records,
            "binding": binding, "scope": "artifact hashes, splits, epoch selection, features, EBM replay, validation metrics",
            "cnn_images_replayed": False, "platt_and_alpha_refitted": False, "test_evaluated": False}

## 3. Sáu baseline nhẹ trên cùng input ngoài fold

EBM dùng nguyên hyperparameter của EBM chuẩn đã train. Trước khi so sánh, fit lại
EBM tham chiếu trên runtime này và yêu cầu tái tạo đúng dự đoán đã lưu; nếu không
khớp thì dừng để tránh trộn nền tảng. Chạy baseline thật trên Kaggle Linux x86.

Logistic dùng logit concept, median imputation, one-hot sex và chuẩn hóa trong
Pipeline. Chọn C bằng 5-fold CV log-loss **trong train**, với preprocessing fit
riêng mỗi fold. Validation chỉ chọn ngưỡng Macro-F1. Không tối ưu trên test.

In [ ]:
def check_analysis_environment(binding):
    require(platform.system() == "Linux" and platform.machine() in ["x86_64", "AMD64"],
            "Run real baseline fitting on Kaggle Linux x86, matching the CNN/EBM training platform.")
    for name in ["interpret-core", "numpy", "scikit-learn", "pandas"]:
        require(importlib.metadata.version(name) == binding["environment"][name],
                "Version mismatch before paired comparison: " + name)

def baseline_inputs(raw, data, group, logit=False):
    pieces, names, types = [], [], []
    if group != "clinical_only":
        probability = raw
        if logit:
            p = np.clip(probability, 1e-6, 1 - 1e-6)
            probability = np.log(p / (1 - p))
        pieces.append(probability)
        names += data["names"][:14]
        types += data["types"][:14]
    if group != "concept_only":
        pieces.append(data["z"])
        names += data["names"][14:]
        types += data["types"][14:]
    return np.column_stack(pieces), names, types

def make_logistic(names, seed, cv_folds=5, candidates=None):
    categorical = [i for i, name in enumerate(names) if name == "clinical_sex_recorded"]
    numeric = [i for i in range(len(names)) if i not in categorical]
    blocks = []
    if numeric:
        blocks.append(("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler())]), numeric))
    if categorical:
        blocks.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical))
    pipeline = Pipeline([("preprocess", ColumnTransformer(blocks)),
                         ("logistic", LogisticRegression(max_iter=5000, random_state=seed))])
    search = GridSearchCV(pipeline, {"logistic__C": np.logspace(-4, 2, 13) if candidates is None else candidates},
                          cv=StratifiedKFold(cv_folds, shuffle=True, random_state=seed),
                          scoring="neg_log_loss", refit=True, n_jobs=1, error_score="raise")
    return search

def paired_bootstrap(y, model, reference, seed, repeats=None):
    repeats = BOOTSTRAP if repeats is None else repeats
    rng = np.random.default_rng(seed)
    functions = {"auroc": roc_auc_score, "auprc": average_precision_score,
                 "brier": brier_score_loss, "logloss": lambda t, p: log_loss(t, p, labels=[0, 1])}
    samples = {name: [] for name in functions}
    for _ in range(repeats):
        rows = rng.integers(0, len(y), len(y))
        if np.unique(y[rows]).size < 2:
            continue
        for name, function in functions.items():
            samples[name].append(function(y[rows], model[rows]) - function(y[rows], reference[rows]))
    return {name: {"model_minus_reference": float(function(y, model) - function(y, reference)),
                   "ci95": np.quantile(samples[name], [0.025, 0.975]).tolist() if samples[name] else None,
                   "valid_resamples": len(samples[name])} for name, function in functions.items()}

def fit_baselines(source, data, audited, folder, logistic_folds=5, logistic_cs=None, enforce_platform=True):
    require(audited["passed"], "A passing native output audit is required.")
    if enforce_platform:
        check_analysis_environment(audited["binding"])
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    require(not (folder / "comparison_validation.json").exists(), "Use a new output directory for a new comparison.")
    train, valid = data["train"], data["valid"]
    y_train, y_valid = data["y"][train].astype(int), data["y"][valid].astype(int)
    template = source.model("uncalibrated_standard_ebm/ebm.joblib")
    parameters = template.get_params()
    predictions, records = {}, []
    # Replay reference first. Never proceed to comparisons on a mismatched runtime.
    replay = ExplainableBoostingClassifier(**parameters).fit(
        np.column_stack([audited["raw"][train], data["z"][train]]), y_train)
    reference = replay.predict_proba(np.column_stack([audited["raw"][valid], data["z"][valid]]))[:, 1]
    require(np.allclose(reference, audited["outcomes"]["uncalibrated_standard_ebm"], atol=1e-10, rtol=0),
            "Refitted reference differs. Do not compare across EBM fitting environments.")
    for head in ["ebm", "logistic"]:
        for group in ["clinical_only", "concept_only", "concept_clinical"]:
            name = group + "_" + head
            x, names, types = baseline_inputs(audited["raw"], data, group, logit=head == "logistic")
            if head == "ebm":
                if group == "concept_clinical":
                    model = replay
                else:
                    params = {**parameters, "feature_names": names, "feature_types": types}
                    model = ExplainableBoostingClassifier(**params).fit(x[train], y_train)
            else:
                model = make_logistic(names, source.seed, logistic_folds, logistic_cs).fit(x[train], y_train)
            p = model.predict_proba(x[valid])[:, 1]
            threshold = choose_threshold(y_valid, p)
            record = {"seed": source.seed, "model": name, "threshold": threshold, **metric_values(y_valid, p, threshold)}
            if head == "logistic":
                record["selected_C"] = float(model.best_params_["logistic__C"])
            joblib.dump(model, folder / (name + ".joblib"))
            np.savez_compressed(folder / (name + "_validation.npz"), rows=valid, probabilities=p)
            predictions[name] = p
            records.append(record)
            print({"seed": source.seed, "completed_baseline": name}, flush=True)
    for row in audited["records"]:
        name = "acc_ecbm_" + row["variant"]
        predictions[name] = audited["outcomes"][row["variant"]]
        records.append({"seed": source.seed, "model": name, **{k: row[k] for k in
                        ["threshold", "auroc", "auprc", "macro_f1", "brier", "logloss"]}})
    intervals = {name: paired_bootstrap(y_valid, p, reference, source.seed) for name, p in predictions.items()
                 if name != "concept_clinical_ebm"}
    rq1 = paired_bootstrap(y_valid, reference, predictions["clinical_only_ebm"], source.seed)
    result = {"records": records, "reference": "concept_clinical_ebm", "paired_vs_reference": intervals,
              "rq1_concept_clinical_minus_clinical": rq1, "test_evaluated": False,
              "platform": {"system": platform.system(), "machine": platform.machine()},
              "concept_input": "raw out-of-fold probabilities; logistic takes logit",
              "reference_refit_max_abs_difference": float(np.max(np.abs(reference -
                   audited["outcomes"]["uncalibrated_standard_ebm"]))),
              "provenance": {"analysis_protocol": "v5-native-analysis-1", "seed": source.seed,
                  "source_complete_sha256": hashlib.sha256(source.read("complete.json")).hexdigest(),
                  "source_binding_sha256": hashlib.sha256(source.read("binding.json")).hexdigest(),
                  "manifest_sha256": data["manifest_hash"], "schema_sha256": data["schema_hash"],
                  "bootstrap_repeats": BOOTSTRAP, "logistic_cv_folds": logistic_folds,
                  "logistic_C_candidates": np.logspace(-4, 2, 13).tolist() if logistic_cs is None else logistic_cs,
                  "environment": {name: importlib.metadata.version(name) for name in
                      ["interpret-core", "numpy", "scikit-learn", "pandas"]},
                  "artifacts": {path.name: file_hash(path) for path in sorted(folder.iterdir()) if path.is_file()}}}
    write_json(folder / "comparison_validation.json", result)
    pd.DataFrame(records).to_csv(folder / "models_validation.csv", index=False)
    return result

## 4. Bảng và hình cho phần Experiments

Gộp đúng các seed thực sự hoàn tất; mean ± SD dùng độ lệch chuẩn mẫu. Bảng
intervention luôn ghi số xác nhận thực tế và số reference còn lại. RQ1 lấy
ΔAUROC (concept+clinical − clinical-only). Các kết luận còn thiếu được ghi rõ;
không biến CI chứa 0 thành bằng chứng hai phương pháp tương đương.

In [ ]:
def aggregate_records(records):
    frame = pd.DataFrame(records)
    require(not frame.empty and not frame.duplicated(["seed", "model"]).any(), "Duplicate or empty model records.")
    metrics = ["auroc", "auprc", "macro_f1", "brier", "logloss"]
    rows = []
    for model, selected in frame.groupby("model", sort=True):
        row = {"model": model, "n_seeds": selected.seed.nunique(),
               "seeds": ",".join(str(s) for s in sorted(selected.seed.unique()))}
        for metric in metrics:
            require(np.isfinite(selected[metric]).all(), "A model metric is missing or non-finite.")
            row[metric + "_mean"] = float(selected[metric].mean())
            row[metric + "_sd"] = float(selected[metric].std(ddof=1)) if row["n_seeds"] > 1 else None
        rows.append(row)
    return pd.DataFrame(rows)

def make_comparison_report(comparisons, sources, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    records = [record for comparison in comparisons for record in comparison["records"]]
    table = aggregate_records(records)
    # The saved raw standard EBM is the same reference, not an independent result.
    table = table[~table.model.eq("acc_ecbm_uncalibrated_standard_ebm")].reset_index(drop=True)
    table.to_csv(output / "models_mean_sd_validation.csv", index=False)
    lines = ["# LungXplain v5 — validation results", "", "Test remains unevaluated.", "",
             "| Model | Seeds | AUROC | AUPRC | Macro-F1 | Brier | Log-loss |",
             "|---|---:|---:|---:|---:|---:|---:|"]
    for row in table.to_dict("records"):
        def formatted(metric):
            mean, sd = row[metric + "_mean"], row[metric + "_sd"]
            return f"{mean:.4f} ± {sd:.4f}" if row["n_seeds"] > 1 else f"{mean:.4f} (one seed)"
        lines.append("| " + " | ".join([row["model"], str(row["n_seeds"])] +
                    [formatted(m) for m in ["auroc", "auprc", "macro_f1", "brier", "logloss"]]) + " |")
    lines += ["", "Standard EBM and intervention-aware EBM are distinct fitted heads;",
              "three intervention strategies share each fitted head. The raw standard",
              "ACC-ECBM reference is reported once as concept_clinical_ebm.", "",
              "## RQ1: concept contribution beyond clinical inputs", ""]
    for comparison in comparisons:
        seed = comparison["records"][0]["seed"]
        result = comparison["rq1_concept_clinical_minus_clinical"]["auroc"]
        low, high = result["ci95"] if result["ci95"] else (None, None)
        decision = "CI entirely above zero" if low is not None and low > 0 else "improvement not established by this CI"
        lines.append(f"- Seed {seed}: ΔAUROC={result['model_minus_reference']:.4f}, CI95={result['ci95']}; {decision}.")
    paired_rows = []
    lines += ["", "## RQ3: unconfirmed concept correction", ""]
    for seed, source in sorted(sources.items()):
        intervals = source.json(MAIN_VARIANT + "/paired_bootstrap_validation.json")
        for count, comparisons_at_count in intervals.items():
            for comparison, metrics_at_count in comparisons_at_count.items():
                for metric, result in metrics_at_count.items():
                    interval = result["ci95"]
                    paired_rows.append({"seed": seed, "requested_concepts": int(count),
                        "comparison": comparison, "metric": metric,
                        "delta_acc_minus_reference": result["delta_acc_minus_reference"],
                        "ci95_low": interval[0] if interval else None,
                        "ci95_high": interval[1] if interval else None,
                        "valid_resamples": result["valid_resamples"]})
        for count in ["1", "2"]:
            result = intervals[count]["acc_vs_hard_correction"]["remaining_concept_logloss"]
            interval = result["ci95"]
            state = "lower log-loss supported by this CI" if interval and interval[1] < 0 else "improvement not established by this CI"
            lines.append(f"- Seed {seed}, m={count}: Δlog-loss={result['delta_acc_minus_reference']}, CI95={interval}; {state}.")
    pd.DataFrame(paired_rows).to_csv(output / "intervention_paired_ci_validation.csv", index=False)
    lines += ["", "## RQ4: outcome after confirmation", "",
              "Per-seed ACC−hard and ACC−automatic AUROC/log-loss estimates and CIs at m=1,2,14",
              "are in intervention_paired_ci_validation.csv. Positive ΔAUROC and negative Δlog-loss",
              "favor ACC. A CI crossing zero does not establish equivalence or non-inferiority."]
    lines += ["", "## Remaining evidence", "", "- CNN black-box and image+clinical late fusion are pending; no values are filled in.",
              "- Validation threshold selection makes these development results, not a final test claim.",
              "- The audit did not rerun CNN images or refit Platt/alpha; retain checkpoints for deeper verification.",
              "- Report-derived simulated confirmation is not an actual clinician intervention.", ""]
    curves = []
    for seed, source in sources.items():
        frame = pd.read_csv(io.BytesIO(source.read(MAIN_VARIANT + "/interventions_validation.csv")))
        frame.insert(0, "seed", seed)
        curves.append(frame)
    curve_table = pd.concat(curves, ignore_index=True)
    curve_table.to_csv(output / "interventions_all_seeds_validation.csv", index=False)
    grouped = curve_table.groupby(["requested_concepts", "strategy"]).agg(
        mean_actual_confirmations=("mean_actual_confirmations", "mean"),
        remaining_reference_count=("remaining_reference_count", "min"),
        auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"),
        concept_logloss_mean=("remaining_concept_logloss", "mean"),
        concept_logloss_sd=("remaining_concept_logloss", "std")).reset_index()
    grouped.to_csv(output / "interventions_mean_sd_validation.csv", index=False)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for strategy, label in [("no_correction", "No correction"), ("hard_correction", "Hard correction"), ("acc", "ACC")]:
        subset = grouped[grouped.strategy.eq(strategy)]
        for axis, name, title in [(axes[0], "concept_logloss", "Unconfirmed concept log-loss"),
                                  (axes[1], "auroc", "Mortality AUROC")]:
            x = subset.requested_concepts.to_numpy()
            mean = subset[name + "_mean"].to_numpy()
            sd = subset[name + "_sd"].to_numpy()
            axis.plot(x, mean, marker="o", ms=3, label=label)
            if len(sources) > 1:
                axis.fill_between(x, mean - sd, mean + sd, alpha=0.15)
            axis.set(title=title, xlabel="Requested confirmations (available references only)")
            axis.grid(alpha=0.2)
    axes[0].legend()
    figure.suptitle("Validation; shading is SD across seeds, not a confidence interval")
    figure.tight_layout()
    figure.savefig(output / "intervention_curves_validation.png", dpi=200)
    figure.savefig(output / "intervention_curves_validation.svg")
    plt.show()
    plt.close(figure)
    (output / "results_validation.md").write_text("\n".join(lines))
    write_json(output / "analysis_status.json", {"completed_seeds": sorted(sources),
        "missing_seeds": sorted(set(SEEDS) - set(sources)), "test_evaluated": False,
        "blackbox_baselines_completed": False, "cnn_retrained": False, "evaluation_split": "valid"})
    return table

## 5. Chạy phân tích khi có output

Nếu chưa gắn output training, cell chỉ ghi trạng thái đang chờ; không tạo metrics
giả. Mỗi seed cần file `complete.json` và artifact khớp hash. ZIP được đọc trực
tiếp, không giải nén checkpoint CNN ra ổ đĩa. Các file model/dự đoán lưu private.

In [ ]:
DATA = read_cohort(INPUT)
RUNS = find_runs(SEARCH)
if not RUNS:
    write_json(OUTPUT / "analysis_status.json", {"state": "awaiting_completed_native_training_output",
        "missing_seeds": SEEDS, "test_evaluated": False, "metrics_created": False})
    print("Chưa có output native hoàn tất. Chờ GPU xong rồi Add Input → Notebook output private.")
else:
    COMPARISONS = []
    for seed, source in sorted(RUNS.items()):
        audited = audit_run(source, DATA)
        write_json(OUTPUT / f"seed{seed}_audit.json", {k: v for k, v in audited.items()
                   if k not in ["raw", "outcomes", "binding"]})
        comparison = fit_baselines(source, DATA, audited, OUTPUT / f"seed{seed}")
        COMPARISONS.append(comparison)
    SUMMARY = make_comparison_report(COMPARISONS, RUNS, OUTPUT)
    display(SUMMARY)
    print("Đã lưu bảng/hình validation và model baseline tại", OUTPUT)